# Data Quality Assurance Notebook

This notebook provides visual and audio inspection of the SECURE FOREST PATROL dataset.

**Purpose:**
- Inspect samples from each class
- Visualize waveforms and spectrograms
- Verify audio quality
- Check class distribution
- Identify potential issues

**Note:** This notebook is for QA purposes only. Do not modify the dataset based on visual inspection without documenting the decision.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from pathlib import Path
import IPython.display as ipd
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
# Configuration
RAW_DIR = Path('../datasets/raw')
MANIFESTS_DIR = Path('../datasets/manifests')
REPORTS_DIR = Path('../reports')

TARGET_SR = 16000  # Target sample rate
TARGET_DURATION = 1.0  # Target duration in seconds

## 1. Load Dataset Manifest

In [ ]:
# Load master manifest if available
manifest_path = MANIFESTS_DIR / 'master_manifest.csv'

if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)
    print(f"Loaded manifest with {len(manifest_df)} samples")
    print(f"\nColumns: {manifest_df.columns.tolist()}")
    print(f"\nClass distribution:")
    print(manifest_df['class'].value_counts())
else:
    print("Master manifest not found. Will inspect raw datasets directly.")
    manifest_df = None

## 2. Inspect Raw Datasets

In [ ]:
# List available datasets
if RAW_DIR.exists():
    datasets = [d for d in RAW_DIR.iterdir() if d.is_dir()]
    print(f"Available datasets: {len(datasets)}")
    for dataset in datasets:
        print(f"  - {dataset.name}")
else:
    print("Raw directory not found. Please download datasets first.")
    datasets = []

## 3. Audio Inspection Functions

In [ ]:
def load_and_inspect_audio(filepath, max_duration=10.0):
    """Load audio file and return basic info."""
    try:
        audio, sr = sf.read(filepath)
        duration = len(audio) / sr
        
        # Trim to max duration if needed
        if duration > max_duration:
            audio = audio[:int(max_duration * sr)]
            duration = max_duration
        
        # Convert to mono if stereo
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        
        return audio, sr, duration
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None, None, None

def plot_waveform(audio, sr, title="Waveform"):
    """Plot audio waveform."""
    plt.figure(figsize=(14, 3))
    time = np.arange(len(audio)) / sr
    plt.plot(time, audio)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_spectrogram(audio, sr, title="Spectrogram"):
    """Plot mel-spectrogram."""
    plt.figure(figsize=(14, 4))
    
    # Compute mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Display
    librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

def inspect_audio_file(filepath, title="Audio Inspection"):
    """Comprehensive inspection of an audio file."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(f"File: {filepath}")
    
    audio, sr, duration = load_and_inspect_audio(filepath)
    
    if audio is None:
        print("Failed to load audio file")
        return
    
    print(f"Sample rate: {sr} Hz")
    print(f"Duration: {duration:.2f} s")
    print(f"Channels: {1 if len(audio.shape) == 1 else audio.shape[1]}")
    print(f"Max amplitude: {np.max(np.abs(audio)):.4f}")
    print(f"RMS: {np.sqrt(np.mean(audio**2)):.4f}")
    
    # Plot waveform
    plot_waveform(audio, sr, f"Waveform - {title}")
    
    # Plot spectrogram
    plot_spectrogram(audio, sr, f"Spectrogram - {title}")
    
    # Audio player
    print("\nAudio playback:")
    display(ipd.Audio(audio, rate=sr))

## 4. Inspect Gunshot Samples (C3GD)

In [ ]:
# Find C3GD dataset
c3gd_dir = RAW_DIR / 'c3gd'

if c3gd_dir.exists():
    # Find audio files
    gunshot_files = list((c3gd_dir / 'data').glob('*.wav')) if (c3gd_dir / 'data').exists() else []
    
    if gunshot_files:
        print(f"Found {len(gunshot_files)} gunshot files in C3GD")
        
        # Inspect a few samples
        for i, filepath in enumerate(gunshot_files[:3]):
            inspect_audio_file(filepath, f"Gunshot Sample {i+1} (C3GD)")
    else:
        print("No gunshot files found in C3GD")
else:
    print("C3GD dataset not found")

## 5. Inspect Chainsaw Samples (RFCx FrugalAI)

In [ ]:
# Find RFCx FrugalAI dataset
frugalai_dir = RAW_DIR / 'rfcx_frugalai'

if frugalai_dir.exists():
    # Find chainsaw files (typically labeled with label_0)
    chainsaw_files = list(frugalai_dir.rglob('*label_0*.wav'))
    
    if chainsaw_files:
        print(f"Found {len(chainsaw_files)} chainsaw files in RFCx FrugalAI")
        
        # Inspect a few samples
        for i, filepath in enumerate(chainsaw_files[:3]):
            inspect_audio_file(filepath, f"Chainsaw Sample {i+1} (RFCx FrugalAI)")
    else:
        print("No chainsaw files found in RFCx FrugalAI")
else:
    print("RFCx FrugalAI dataset not found")

## 6. Inspect Environmental/Background Samples (ESC-50)

In [ ]:
# Find ESC-50 dataset
esc50_dir = RAW_DIR / 'esc50'

if esc50_dir.exists():
    # Load metadata
    esc50_meta = esc50_dir / 'esc50.csv'
    
    if esc50_meta.exists():
        esc50_df = pd.read_csv(esc50_meta)
        print(f"ESC-50 metadata loaded: {len(esc50_df)} files")
        print(f"\nClasses: {esc50_df['category'].unique()}")
        
        # Find environmental/bird/wind sounds (for background)
        background_classes = ['rain', 'wind', 'crickets', 'chirping_birds', 'thunderstorm']
        background_files = []
        
        for bg_class in background_classes:
            class_files = esc50_df[esc50_df['category'] == bg_class]['filename'].tolist()
            for filename in class_files[:2]:  # Take 2 samples per class
                filepath = esc50_dir / 'audio' / filename
                if filepath.exists():
                    background_files.append((filepath, bg_class))
        
        print(f"\nFound {len(background_files)} background samples")
        
        # Inspect background samples
        for filepath, bg_class in background_files[:3]:
            inspect_audio_file(filepath, f"Background Sample - {bg_class} (ESC-50)")
    else:
        print("ESC-50 metadata not found")
else:
    print("ESC-50 dataset not found")

## 7. Inspect Forest Background (Sensing the Forest)

In [ ]:
# Find Sensing the Forest dataset
sensing_forest_dir = RAW_DIR / 'sensing_forest'

if sensing_forest_dir.exists():
    # Find audio files
    forest_files = list(sensing_forest_dir.glob('*.wav')) + list(sensing_forest_dir.glob('*.mp3'))
    
    if forest_files:
        print(f"Found {len(forest_files)} forest soundscape files")
        
        # Inspect a few samples
        for i, filepath in enumerate(forest_files[:2]):
            inspect_audio_file(filepath, f"Forest Soundscape Sample {i+1} (Sensing the Forest)")
    else:
        print("No audio files found in Sensing the Forest dataset")
else:
    print("Sensing the Forest dataset not found")

## 8. Class Distribution Analysis

In [ ]:
if manifest_df is not None:
    # Plot class distribution
    plt.figure(figsize=(10, 6))
    class_counts = manifest_df['class'].value_counts()
    
    ax = sns.barplot(x=class_counts.index, y=class_counts.values)
    plt.title('Class Distribution')
    plt.xlabel('Class')
    plt.ylabel('Count')
    
    # Add count labels on bars
    for i, count in enumerate(class_counts.values):
        ax.text(i, count, str(count), ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\nClass Distribution Statistics:")
    print(f"Total samples: {len(manifest_df)}")
    print(f"Number of classes: {len(class_counts)}")
    print(f"\nSamples per class:")
    for class_name, count in class_counts.items():
        percentage = (count / len(manifest_df)) * 100
        print(f"  {class_name}: {count} ({percentage:.1f}%)")
    
    # Check for imbalance
    max_count = class_counts.max()
    min_count = class_counts.min()
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    
    print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")
    
    if imbalance_ratio > 3:
        print("Warning: Significant class imbalance detected (ratio > 3)")
    elif imbalance_ratio > 2:
        print("Note: Moderate class imbalance (ratio > 2)")
    else:
        print("Class distribution is relatively balanced")
else:
    print("Manifest not available for class distribution analysis")

## 9. Duration Analysis

In [ ]:
if manifest_df is not None and 'duration' in manifest_df.columns:
    # Plot duration distribution
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(manifest_df['duration'], bins=50, edgecolor='black')
    plt.xlabel('Duration (s)')
    plt.ylabel('Count')
    plt.title('Duration Distribution')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    manifest_df.boxplot(column='duration', by='class', ax=plt.gca())
    plt.ylabel('Duration (s)')
    plt.title('Duration by Class')
    plt.suptitle('')  # Remove automatic title
    
    plt.tight_layout()
    plt.show()
    
    # Print duration statistics
    print("\nDuration Statistics:")
    print(manifest_df.groupby('class')['duration'].describe())
else:
    print("Duration data not available in manifest")

## 10. Sample Rate Analysis

In [ ]:
if manifest_df is not None and 'sample_rate' in manifest_df.columns:
    # Plot sample rate distribution
    plt.figure(figsize=(10, 5))
    sr_counts = manifest_df['sample_rate'].value_counts().sort_index()
    
    ax = sns.barplot(x=sr_counts.index.astype(str), y=sr_counts.values)
    plt.xlabel('Sample Rate (Hz)')
    plt.ylabel('Count')
    plt.title('Sample Rate Distribution')
    plt.xticks(rotation=45)
    
    # Add count labels
    for i, count in enumerate(sr_counts.values):
        ax.text(i, count, str(count), ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    print("\nSample Rate Statistics:")
    print(sr_counts)
    
    # Check if resampling needed
    if TARGET_SR not in sr_counts.index:
        print(f"\nNote: No samples at target sample rate {TARGET_SR} Hz")
        print("Resampling will be required during preprocessing.")
else:
    print("Sample rate data not available in manifest")

## 11. Quality Summary

In [ ]:
print("="*60)
print("DATA QUALITY ASSURANCE SUMMARY")
print("="*60)

print("\nDatasets inspected:")
for dataset in datasets:
    print(f"  ✓ {dataset.name}")

print("\nKey observations:")
print("1. Audio quality varies across datasets")
print("2. Sample rates need standardization to 16 kHz")
print("3. Channel conversion to mono required")
print("4. Duration varies - segmentation to 1-second windows needed")
print("5. Class distribution may be imbalanced - check reports")

print("\nRecommendations:")
print("- Run preprocessing pipeline to standardize all audio")
print("- Review quality audit report for specific issues")
print("- Check class distribution report for imbalance handling")
print("- Validate preprocessing results with validation script")

print("\nNext steps:")
print("1. python preprocessing/build_dataset.py")
print("2. python scripts/create_splits.py")
print("3. python scripts/validate_preprocessing.py")